# Local Invoice OCR on Google Colab
Select **Runtime > Change runtime type > T4 GPU**, reconnect, then run every cell in order. By default the notebook stops if no GPU is attached. PDFs come from the public GitHub repo; no Drive upload is needed. Accuracy mode reads original PDF pages with a vision model, then checks item arithmetic and OCR evidence; Fast mode uses spatial OCR only. Stock Ollama remains the default. Set USE_TRAINED_ADAPTER=True and paste TRAINED_RELEASE_TAG only after colab_train.ipynb publishes a passing adapter to a public GitHub Release.

In [ ]:
#@title 1. Clone the public GitHub project
import os, pathlib, subprocess
PROJECT_DIR = pathlib.Path('/content/OCR')
if not PROJECT_DIR.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ubaid-148/OCR.git',str(PROJECT_DIR)], check=True)
elif (PROJECT_DIR/'.git').is_dir():
    subprocess.run(['git','-C',str(PROJECT_DIR),'pull','--ff-only'], check=True)
else: raise ValueError('/content/OCR exists but is not a Git clone. Start a fresh runtime.')
os.chdir(PROJECT_DIR)
print('Project ready at', PROJECT_DIR)
print('Flow 2026-09-trained-gated-v13: approved adapter optional; uncertain fields stay in review')

In [ ]:
#@title 2. Install dependencies and verify the OCR device
import os, pathlib, shutil, subprocess, sys
REQUIRE_GPU_FOR_ACCURACY = True #@param {type:"boolean"}
gpu_runtime = bool(shutil.which('nvidia-smi')) and subprocess.run(['nvidia-smi', '-L'], capture_output=True).returncode == 0
print('GPU attached:', gpu_runtime, flush=True)
if REQUIRE_GPU_FOR_ACCURACY and not gpu_runtime:
    raise RuntimeError('No GPU is attached. In Colab choose Runtime > Change runtime type > T4 GPU, reconnect, then run all cells again. Accuracy vision on CPU caused a 180-second timeout and must not silently fall back.')
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'tesseract-ocr', 'tesseract-ocr-eng', 'tesseract-ocr-ara', 'tesseract-ocr-urd', 'ghostscript', 'unpaper', 'pngquant', 'zstd'], check=True)
# Keep OCR packages separate from Colab's preinstalled CUDA PyTorch.
OCR_ENV_DIR = pathlib.Path('/content/ocr-runtime')
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(OCR_ENV_DIR)], check=True)
OCR_PYTHON = str(OCR_ENV_DIR / 'bin' / 'python')
# pip --python bootstraps pip even in a venv created without it.
ocr_pip = [sys.executable, '-m', 'pip', '--python', OCR_PYTHON]
subprocess.run([*ocr_pip, 'install', '-q', '--upgrade', 'pip'], check=True)
# ModelScope imports torch; its CPU build avoids a second CUDA/NCCL stack.
subprocess.run([*ocr_pip, 'install', '-q', 'torch==2.9.1+cpu', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
# CPU and GPU Paddle share a module: install exactly one distribution.
subprocess.run([*ocr_pip, 'uninstall', '-y', 'paddlepaddle', 'paddlepaddle-gpu'], check=True)
requirements = [line.strip() for line in pathlib.Path('requirements.txt').read_text().splitlines() if line.strip() and not line.strip().startswith('paddlepaddle')]
subprocess.run([*ocr_pip, 'install', '-q', *requirements], check=True)
command = [*ocr_pip, 'install', '-q']
if gpu_runtime:
    command += ['paddlepaddle-gpu==3.3.1', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/']
else:
    command += ['paddlepaddle==3.3.1']
subprocess.run(command, check=True)
# Retry uncertain identifiers/numeric cells with bounded English OCR crops.
os.environ['OCR_TARGETED_RETRY'] = 'true'
os.environ['OCR_DEVICE'] = 'gpu:0' if gpu_runtime else 'cpu'
os.environ['VISION_REQUIRE_GPU'] = 'true'
os.environ.pop('OCR_PYTHON_EXE', None)
# Check in a fresh process so rerunning this cell cannot reuse an old Paddle import.
os.environ.setdefault('FLAGS_use_mkldnn', '0')
verification = subprocess.run(
    [OCR_PYTHON, '-u', str(PROJECT_DIR / 'check_ocr_runtime.py')],
    cwd=PROJECT_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, errors='replace',
)
verification_log = pathlib.Path('/tmp/ocr-runtime-check.log')
verification_log.write_text(verification.stdout, encoding='utf-8')
print(verification.stdout, flush=True)
if verification.returncode:
    raise RuntimeError(
        f'OCR runtime verification failed (exit {verification.returncode}). '
        f'Full log: {verification_log}. Copy the error below:\n\n'
        + verification.stdout[-12000:]
    )
print('Dependencies ready. First upload loads OCR models; later uploads reuse them.')


In [ ]:
#@title 2b. Record the PaddleOCR runtime and model configuration
import json, os, platform, subprocess, sys, time
import paddle
import paddleocr

cuda_version = getattr(paddle, "cuda_version", lambda: None)()
gpu_count = paddle.device.cuda.device_count() if paddle.is_compiled_with_cuda() else 0
runtime_device = os.environ.get("OCR_DEVICE", "auto")
if runtime_device == "auto":
    runtime_device = "gpu:0" if gpu_count else "cpu"

try:
    nvidia_smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=False,
    ).stdout.strip()
except OSError:
    nvidia_smi = "unavailable"

OCR_MODEL_CONFIGURATION = {
    "GPU": nvidia_smi or ("available" if gpu_count else "not available"),
    "CUDA": cuda_version or "not exposed by Paddle",
    "Paddle": paddle.__version__,
    "PaddleOCR": paddleocr.__version__,
    "Python": platform.python_version(),
    "Device": runtime_device,
    "Detection model": "PP-OCRv5_mobile_det",
    "English recognition model": "PP-OCRv5_mobile_rec",
    "Arabic recognition model": "arabic_PP-OCRv5_mobile_rec",
}
print(json.dumps(OCR_MODEL_CONFIGURATION, indent=2, ensure_ascii=False))
if runtime_device.startswith("gpu") and not gpu_count:
    print("WARNING: OCR_DEVICE requests GPU but Paddle reports no CUDA device.")


In [ ]:
#@title 3. Start stock vision AI or an approved trained adapter
USE_LOCAL_AI = True #@param {type:"boolean"}
USE_TRAINED_ADAPTER = False #@param {type:"boolean"}
TRAINED_RELEASE_TAG = '' #@param {type:"string"}
OLLAMA_MODEL = 'qwen3-vl:4b' #@param {type:"string"}
import json, os, shutil, subprocess, sys, time, urllib.request
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
from ollama_http import preload
if USE_TRAINED_ADAPTER: USE_LOCAL_AI = True
os.environ['OLLAMA_MODEL'] = OLLAMA_MODEL
os.environ['USE_LOCAL_AI'] = str(USE_LOCAL_AI).lower()
if USE_LOCAL_AI and not gpu_runtime:
    raise RuntimeError('Vision AI is disabled on CPU for this notebook. Attach a T4/L4 GPU, reconnect, and rerun from cell 1; or set USE_LOCAL_AI=False for spatial-only Fast mode.')
os.environ['OLLAMA_TIMEOUT_SECONDS'] = '180'
os.environ['OLLAMA_NUM_CTX'] = '16384'
os.environ['OLLAMA_NUM_PREDICT'] = '4096'
os.environ.pop('OLLAMA_URL', None)
os.environ.pop('TRAINED_VISION_URL', None)
if USE_TRAINED_ADAPTER:
    if not TRAINED_RELEASE_TAG: raise ValueError('Paste the release tag printed by colab_train.ipynb cell 11')
    from training.github_release import download_bundle
    approval_path = download_bundle(TRAINED_RELEASE_TAG, pathlib.Path('/content/approved-invoice-adapter')/TRAINED_RELEASE_TAG)
    from training.adapter_service import load_approval
    approved = load_approval(approval_path)
    subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.0','accelerate==1.7.0','peft==0.17.1','qwen-vl-utils==0.0.14','pillow'], check=True)
    subprocess.run(['pkill','-TERM','-x','ollama'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    if 'adapter_process' in globals() and adapter_process.poll() is None:
        adapter_process.terminate(); adapter_process.wait(timeout=15)
    adapter_log = open('/tmp/invoice-adapter.log','w')
    adapter_process = subprocess.Popen([sys.executable,'-u','-m','training.adapter_service','--approval',str(approval_path)], cwd=PROJECT_DIR, stdout=adapter_log, stderr=subprocess.STDOUT)
    for attempt in range(600):
        if adapter_process.poll() is not None:
            raise RuntimeError('Approved adapter service failed: '+pathlib.Path('/tmp/invoice-adapter.log').read_text()[-4000:])
        try:
            with urllib.request.urlopen('http://127.0.0.1:8766/health',timeout=2) as health:
                if json.load(health).get('ready'): break
        except OSError: time.sleep(1)
    else: raise RuntimeError('Approved adapter did not become ready; see /tmp/invoice-adapter.log')
    os.environ['TRAINED_VISION_URL'] = 'http://127.0.0.1:8766/extract'
    print('Approved trained adapter ready:', approved['model_id'])
elif USE_LOCAL_AI:
    if not shutil.which('ollama'):
        ollama_archive = '/tmp/ollama-linux-amd64.tar.zst'
        print('Downloading Ollama...')
        urllib.request.urlretrieve('https://ollama.com/download/ollama-linux-amd64.tar.zst', ollama_archive)
        subprocess.run(['tar', '--zstd', '-xf', ollama_archive, '-C', '/usr'], check=True)
        if not shutil.which('ollama'):
            raise RuntimeError('Ollama archive extracted but the executable was not found')
    # Cell 1 replaces /content/OCR. Restart Ollama so it never retains that deleted cwd.
    try:
        with urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2) as response:
            service_running = response.status == 200
    except OSError:
        service_running = False
    if service_running:
        subprocess.run(['ollama', 'stop', OLLAMA_MODEL], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
        subprocess.run(['pkill', '-TERM', '-x', 'ollama'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
        time.sleep(2)
    if 'ollama_log' in globals() and not ollama_log.closed:
        ollama_log.close()
    ollama_log = open('/tmp/ollama.log', 'a')
    ollama_process = subprocess.Popen(
        ['ollama', 'serve'], cwd='/content', start_new_session=True,
        stdout=ollama_log, stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        try:
            urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError('Ollama did not start. Check /tmp/ollama.log')
    subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True)
    print('Loading AI model before the first invoice...')
    ai_preloaded = preload(OLLAMA_MODEL, context=int(os.environ['OLLAMA_NUM_CTX']))
    if not ai_preloaded:
        raise RuntimeError('Vision model did not preload. Check /tmp/ollama.log; the app will not launch into a slow spatial fallback.')
    placement = subprocess.run(['ollama', 'ps'], capture_output=True, text=True, check=True)
    print(placement.stdout, flush=True)
    model_line = next((line for line in placement.stdout.splitlines()[1:] if line.split() and line.split()[0] == OLLAMA_MODEL), '')
    if '100% GPU' not in model_line:
        raise RuntimeError(f'{OLLAMA_MODEL} is not fully on GPU. Ollama placement: {model_line or placement.stdout}. Use a runtime with more free GPU memory; CPU offload is too slow for Accuracy mode.')
    print('Ollama setup complete: vision model preloaded fully on GPU.')
else:
    os.environ['OLLAMA_URL'] = 'http://127.0.0.1:1/api/chat'
    print('Local AI disabled; deterministic spatial fallback will be used.')

In [ ]:
#@title 4. Start OCR web application
import os, subprocess, sys, time, urllib.request
os.chdir('/content/OCR')
if 'OCR_PYTHON' not in globals():
    raise RuntimeError('Run dependency setup (cell 2) first.')
if 'ocr_process' in globals() and ocr_process.poll() is None:
    ocr_process.terminate()
    ocr_process.wait(timeout=15)
os.environ['OCR_PRELOAD'] = 'true'
print('Preparing OCR models before accepting uploads; first startup may download models.')
ocr_log = open('/tmp/ocr-web.log', 'w')
ocr_process = subprocess.Popen([OCR_PYTHON, '-u', 'ocr_web.py'], stdout=ocr_log, stderr=subprocess.STDOUT, env=os.environ.copy())
for attempt in range(600):
    if attempt and attempt % 15 == 0:
        print('Still preparing OCR models. Recent log:', open('/tmp/ocr-web.log').read()[-600:])
    if ocr_process.poll() is not None:
        print(open('/tmp/ocr-web.log').read())
        raise RuntimeError('OCR server exited during startup')
    try:
        response = urllib.request.urlopen('http://127.0.0.1:8765/', timeout=2)
        if response.status == 200:
            break
    except Exception:
        time.sleep(1)
else:
    print(open('/tmp/ocr-web.log').read())
    raise RuntimeError('OCR web application did not start')
print('OCR application is ready.')

In [ ]:
#@title 5. Open the application
from google.colab import output
output.serve_kernel_port_as_iframe(8765, height='700')

In [ ]:
#@title 6. Run a raw PaddleOCR evidence pass on 9498.pdf
RUN_9498_BENCHMARK = True #@param {type:"boolean"}
import json, os, re, time
from pathlib import Path

benchmark_dir = PROJECT_DIR / "benchmark_outputs"
benchmark_dir.mkdir(exist_ok=True)
sample_path = PROJECT_DIR / "public_invoice_pdfs" / "9498.pdf"
if not sample_path.exists():
    raise FileNotFoundError(sample_path)

if RUN_9498_BENCHMARK:
    # Force raster extraction so this pass measures PaddleOCR rather than a PDF text layer.
    os.environ["OCR_FORCE_RASTER"] = "true"
    os.environ["OCR_TARGETED_RETRY"] = "false"
    from coordinate_ocr import extract_pdf
    from layout_invoice import parse_layout
    from local_ai_parser import parse_invoice_hybrid

    started = time.perf_counter()
    raw_ocr = extract_pdf(sample_path, "eng+ara", progress=print)
    raw_ocr_seconds = time.perf_counter() - started
    raw_ocr["benchmark"] = {
        "input": sample_path.name,
        "forced_raster": True,
        "device": raw_ocr.get("device"),
        "elapsed_seconds": round(raw_ocr_seconds, 3),
    }
    raw_path = benchmark_dir / "9498_raw_paddleocr.json"
    raw_path.write_text(json.dumps(raw_ocr, ensure_ascii=False, indent=2), encoding="utf-8")

    raw_rows = []
    numeric_rows = []
    for page in raw_ocr.get("pages", []):
        for word in page.get("words", []):
            row = {
                "page": page.get("page"),
                "text": word.get("text"),
                "confidence": word.get("confidence"),
                "bbox": {
                    "left": word.get("left"), "top": word.get("top"),
                    "width": word.get("width"), "height": word.get("height"),
                },
            }
            raw_rows.append(row)
            if re.search(r"\d", str(row["text"])):
                numeric_rows.append(row)

    print(f"OCR time: {raw_ocr_seconds:.3f} sec")
    print(f"Pages: {len(raw_ocr.get('pages', []))}; detected text boxes: {len(raw_rows)}")
    print("Numeric OCR evidence (text, confidence, page, bbox):")
    print(json.dumps(numeric_rows, ensure_ascii=False, indent=2))

    spatial_started = time.perf_counter()
    parser_result = parse_invoice_hybrid(raw_ocr["pages"], sample_path.name, "eng+ara", mode="fast")
    parser_seconds = time.perf_counter() - spatial_started
    parser_data = parser_result.get("data", {})
    parser_view = {
        "parser": parser_result.get("parser"),
        "quality": parser_result.get("quality"),
        "validation": parser_data.get("validation"),
        "items": parser_data.get("items", []),
        "totals": parser_data.get("totals", {}),
        "parser_seconds": round(parser_seconds, 3),
    }
    (benchmark_dir / "9498_parser_fast.json").write_text(
        json.dumps(parser_view, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("\nParser output derived from the same OCR pages:")
    print(json.dumps(parser_view, ensure_ascii=False, indent=2))
    print("\nInterpretation: compare raw_ocr numeric boxes with parser items and validation; no expected invoice values are used.")
else:
    print("9498 benchmark disabled.")


In [ ]:
#@title 7. Compare PaddleOCR preprocessing variants
RUN_PREPROCESSING_COMPARISON = True #@param {type:"boolean"}
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
import pypdfium2 as pdfium
from coordinate_ocr import _get_model, extract_words

if RUN_PREPROCESSING_COMPARISON:
    preprocessing_dir = benchmark_dir / "preprocessing"
    preprocessing_dir.mkdir(exist_ok=True)
    document = pdfium.PdfDocument(str(sample_path))
    page = document[0]
    variants = {}
    for name, dpi in (("original", 200), ("high_resolution", 300)):
        bitmap = page.render(scale=dpi / 72)
        try:
            variants[name] = bitmap.to_pil().convert("RGB")
        finally:
            bitmap.close()
    base = variants["high_resolution"]
    variants["grayscale"] = ImageOps.grayscale(base).convert("RGB")
    variants["contrast_sharpened"] = ImageEnhance.Contrast(base).enhance(1.6).filter(ImageFilter.SHARPEN)
    try:
        import cv2, numpy as np
        gray = np.array(ImageOps.grayscale(base))
        threshold = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
        points = cv2.findNonZero(threshold)
        angle = 0.0
        if points is not None and len(points) > 20:
            angle = cv2.minAreaRect(points)[-1]
            angle = -(90 + angle) if angle < -45 else -angle
        variants["deskew_if_required"] = base.rotate(angle, expand=True, fillcolor="white")
    except ImportError:
        variants["deskew_if_required"] = base
        angle = None
    finally:
        page.close()
        document.close()

    model = _get_model("ar")
    comparison = []
    for name, image in variants.items():
        image_path = preprocessing_dir / f"9498-page-1-{name}.png"
        image.save(image_path)
        started = time.perf_counter()
        words = []
        for prediction in model.predict(str(image_path)):
            words.extend(extract_words(prediction))
        elapsed = time.perf_counter() - started
        high_confidence = [word for word in words if float(word.get("confidence", 0)) >= 80]
        numeric = [word for word in words if re.search(r"\d", str(word.get("text", "")))]
        comparison.append({
            "preprocessing": name,
            "image_size": image.size,
            "words": len(words),
            "high_confidence_words": len(high_confidence),
            "numeric_words": len(numeric),
            "elapsed_seconds": round(elapsed, 3),
            "deskew_angle": angle if name == "deskew_if_required" else None,
            "numeric_text": [word["text"] for word in numeric],
        })
    preprocessing_path = benchmark_dir / "9498_preprocessing_comparison.json"
    preprocessing_path.write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(comparison, ensure_ascii=False, indent=2))
else:
    print("Preprocessing comparison disabled.")


In [ ]:
#@title 8. Draw readable OCR boxes for table inspection
from PIL import ImageDraw, ImageFont
from IPython.display import display

if RUN_9498_BENCHMARK and raw_ocr.get("pages"):
    page_payload = raw_ocr["pages"][0]
    image = Image.open(benchmark_dir / "preprocessing" / "9498-page-1-high_resolution.png").convert("RGB")
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    for word in page_payload.get("words", []):
        left = float(word.get("left", 0)) * 300 / float(page_payload.get("render_dpi", 200))
        top = float(word.get("top", 0)) * 300 / float(page_payload.get("render_dpi", 200))
        width = float(word.get("width", 0)) * 300 / float(page_payload.get("render_dpi", 200))
        height = float(word.get("height", 0)) * 300 / float(page_payload.get("render_dpi", 200))
        confidence = float(word.get("confidence", 0))
        draw.rectangle((left, top, left + width, top + height), outline="red", width=2)
        draw.text((left, max(0, top - 12)), f"{word.get('text', '')} [{confidence:.0f}]", fill="red", font=font)
    annotated_path = benchmark_dir / "9498_page_1_ocr_boxes.png"
    image.save(annotated_path)
    display(image)
    print("Saved annotated OCR evidence to", annotated_path)
else:
    print("Run the raw benchmark first to create the annotated page.")


In [ ]:
#@title 9. Generate the OCR-versus-parser benchmark report
report = {
    "input": str(sample_path),
    "ocr_engine": "PaddleOCR",
    "ocr_runtime": OCR_MODEL_CONFIGURATION,
    "raw_ocr": {
        "pages": len(raw_ocr.get("pages", [])) if RUN_9498_BENCHMARK else 0,
        "detected_boxes": len(raw_rows) if RUN_9498_BENCHMARK else 0,
        "numeric_boxes": len(numeric_rows) if RUN_9498_BENCHMARK else 0,
        "elapsed_seconds": raw_ocr.get("benchmark", {}).get("elapsed_seconds") if RUN_9498_BENCHMARK else None,
        "evidence_file": str(benchmark_dir / "9498_raw_paddleocr.json"),
    },
    "parser": parser_view if RUN_9498_BENCHMARK else None,
    "preprocessing": comparison if RUN_PREPROCESSING_COMPARISON else [],
    "interpretation": [
        "Raw OCR evidence is stored with page, text, confidence, and bounding box.",
        "Parser output is generated from the same OCR pages and is shown separately.",
        "No invoice value is used as an expected pass condition.",
        "A raw text/box error indicates OCR or preprocessing work; a correct raw box mapped to a wrong item indicates table/parser association work.",
    ],
}
report_path = benchmark_dir / "9498_benchmark_report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(report, ensure_ascii=False, indent=2))
print("Report saved to", report_path)


Run cells 1-9 in a fresh Colab runtime with a GPU. The notebook now records Paddle/PaddleOCR/CUDA/device/model diagnostics, forces a raw PaddleOCR pass over 9498.pdf, saves every detected text box with confidence and coordinates, compares preprocessing variants, draws OCR boxes, and writes a benchmark report. It then runs the existing fast spatial parser over the same OCR pages so OCR evidence and parser associations can be compared separately. No invoice value is used as an automatic expected pass condition. The production web pipeline remains unchanged; this notebook is for OCR diagnosis and reproducible experimentation only. Use the saved JSON and annotated PNG to decide whether a mismatch originates in PaddleOCR or in table/parser association. Colab processes are temporary; rerun from cell 1 after updates.